# **Configuración experimental**
Para realizar este paso en la metodología de validación, se utilizará una partición de datos usando un 80% para los datos de entrenamiento y 20% para los datos de prueba, además se garantizará que los hiperparámetros seleccionados sean óptimos y se evitará el sobreajuste, aplicando la validación cruzada (Cross Validation) en el conjunto de entrenamiento.
Adicionalmente, en el tratamiento de valores no disponibles (NA), se adjudica por mediana para las variables numéricas como la altitud y la moda para las variables categóricas.

In [22]:
#CARGA DEL DATASET

# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "arabica_data_cleaned.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "volpatto/coffee-quality-database-from-cqi",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

#print("First 5 records:", df.head())

#Validar datos
#print(df.to_string())

#Filas - Columnas
n_muestras = df.shape[0]
n_caracteristicas = df.shape[1]

print("Número de muestras:", n_muestras)
print("Número de características:", n_caracteristicas)
#print(df.columns)



Using Colab cache for faster access to the 'coffee-quality-database-from-cqi' dataset.
Número de muestras: 1311
Número de características: 44


# **Preparación de los Datos**

In [25]:
#import pandas as pd
#import numpy as np
#from sklearn.model_selection import train_test_split
#from sklearn.preprocessing import StandardScaler

# 1. Estandarización de nombres (Solución al KeyError)
df.columns = [c.replace(' ', '.') for c in df.columns]

# 2. Selección de variables (Asegúrate de que los puntos coincidan)
features_num = ['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body',
                'Balance', 'Uniformity', 'Clean.Cup', 'Sweetness', 'Moisture']
target = 'Total.Cup.Points'

# 3. Manejo de valores nulos (Imputación por mediana)
for col in features_num:
    df[col] = df[col].fillna(df[col].median())

# 4. Definición de X y y
X = df[features_num]
y = df[target]

# 5. División del dataset (Hold-out 80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Escalado de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos listos para entrenamiento. Columnas seleccionadas con éxito.")

Datos listos para entrenamiento. Columnas seleccionadas con éxito.


# **Regresión Lineal**

In [41]:
#Paramétrico	Regresión Lineal

from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
# Configuración del modelo: Regresión Lineal mediante Gradiente Descendente Estocástico
# Usamos 'squared_error' para Regresión Lineal tradicional
modelo_parametrico = SGDRegressor(
    loss='squared_error',
    penalty='l2',          # Regularización Ridge para evitar sobreajuste
    alpha=0.0001,          # Fuerza de la regularización
    learning_rate='constant',
    eta0=0.01,             # Tasa de aprendizaje inicial
    max_iter=1000,
    random_state=42
)

# Entrenamiento
modelo_parametrico.fit(X_train_scaled, y_train)

# Predicción y Evaluación
y_pred_param = modelo_parametrico.predict(X_test_scaled)
mse_param = mean_squared_error(y_test, y_pred_param)
rmse_param = np.sqrt(mse_param)
r2_param = r2_score(y_test, y_pred_param)

print("--- Modelo Paramétrico (Regresión Lineal SGD) ---")
print(f"RMSE: {rmse_param:.4f}")
print(f"R2 Score: {r2_param:.4f}")

--- Modelo Paramétrico (Regresión Lineal SGD) ---
RMSE: 0.3300
R2 Score: 0.9826


In [30]:
#Paramétrico	Regresión Lineal
#No Paramétrico	K-Nearest Neighbors (KNN)
#Ensamble	Random Forest
#Red Neuronal	MLP Regressor
#Vectores de Soporte	SVR

# **Resultados**